Prediction with trained models - We will do both frozen and unfrozen models

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

2025-03-03 13:33:50.845408: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-03 13:33:52.030463: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-03 13:33:52.030519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-03 13:33:52.239166: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-03 13:33:52.592133: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# locate the test data
# names of the test blocks - but should we do the predictions separately for all the rest of the blocks?

main_data_folder = "All_data"
test_blocks = ['block_0103', 'block_0104', 'block_0105', 'block_0106', 'block_0201', 'block_0202', 'block_0205', 'block_0206',
              'block_0302', 'block_0303', 'block_0304', 'block_0305', 'block_0306']

In [3]:
# load the models?

In [4]:
os.listdir('models')

['pre_trained_CNN_architecture_CNN_LSTM_finetuned.keras',
 'pre_trained_CNN_architecture_CNN_LSTM_frozen.keras']

In [5]:
frozen_model = tf.keras.models.load_model('models/pre_trained_CNN_architecture_CNN_LSTM_frozen.keras')

2025-03-03 13:34:56.109891: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


In [6]:
finetuned_model = tf.keras.models.load_model('models/pre_trained_CNN_architecture_CNN_LSTM_finetuned.keras')

In [7]:
# we just need the feature maps to be passed to the loaded models to get the predicted values - ecah is going to give us 7 values. We need to write a function that will allow for the muliple counts and give use the correct predicted values so that e can compare them against the true values.

In [8]:
# consider the first test block

block 0103

In [9]:
file_path = 'All_data/block_0103'

In [10]:
os.listdir(file_path)

['all_np_files',
 '.ipynb_checkpoints',
 'density_maps_seqs_0103.npy',
 'subwindow_seqs_0103.npy']

In [11]:
# we only need the subwindow features

In [12]:
block_0103_test_data = np.load(os.path.join(file_path, "subwindow_seqs_0103.npy"))

In [13]:
block_0103_test_data.shape

(12288, 13, 32, 32, 3)

In [14]:
# get preds from the frozen mdoel
block_0103_preds = frozen_model.predict(block_0103_test_data)

2025-03-03 13:35:11.420251: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 5s 5ms/step


In [15]:
block_0103_preds.shape

(12288, 7)

In [16]:
# Okay, we need the posthoc normalizing code, let's figureout how to do this

In [17]:
# we will later write a function that would help get the final predictions in an easier manner for the rest of the blocks.

In [18]:
# Function to get the predictions given model and image - This is from earlier


# Since we know that all our test images have teh same dimensions, we will get it from a single file we have stored somewhere

file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

In [19]:
loaded_test_image = np.load(file_path)

In [20]:
loaded_test_image.shape

(768, 1024, 3)

In [21]:
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]

In [22]:
image_height, image_width

(768, 1024)

In [23]:
# test_preds = block_0103_preds[:,6]

In [24]:
# test_preds.shape

In [25]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    
    # all_test_sub_windows = np.load(v_stack_folder + "/"+ selected_file)

    # # now, to get the predictions, pass the sub windows
    # test_image_prediction = model.predict(all_test_sub_windows, batch_size = 100)
    
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [26]:
# try_1 = prediction_on_test_data(test_preds, image_height, image_width, stride = 8, kernel_size = 32)

In [27]:
# try_1[0]

In [28]:
# get the prediccted values in a loop

In [29]:
block_0103_preds.shape

(12288, 7)

In [30]:
frozen_preds_0103 = []
for i in range(7):
    test_preds = block_0103_preds[:,i]
    final_preds = prediction_on_test_data(test_preds, image_height, image_width, stride = 8, kernel_size = 32)
    frozen_preds_0103.append(final_preds[0])

In [31]:
frozen_preds_0103

[32.15915139786005,
 66.41855640358008,
 83.4097794788868,
 58.29196432335046,
 43.084299861057225,
 46.19095202510261,
 21.408111328956814]

In [32]:
# Where do we have the true values?
true_val_location = 'All_data/test_true_counts'

In [33]:
all_true_csv_files = [file for file in os.listdir(true_val_location) if file.split('.')[-1] =='csv']
all_true_csv_files.sort()

In [34]:
all_true_csv_files

['true_counts_blk_0101.csv',
 'true_counts_blk_0102.csv',
 'true_counts_blk_0103.csv',
 'true_counts_blk_0104.csv',
 'true_counts_blk_0105.csv',
 'true_counts_blk_0106.csv',
 'true_counts_blk_0201.csv',
 'true_counts_blk_0202.csv',
 'true_counts_blk_0203.csv',
 'true_counts_blk_0204.csv',
 'true_counts_blk_0205.csv',
 'true_counts_blk_0206.csv',
 'true_counts_blk_0301.csv',
 'true_counts_blk_0302.csv',
 'true_counts_blk_0303.csv',
 'true_counts_blk_0304.csv',
 'true_counts_blk_0305.csv',
 'true_counts_blk_0306.csv']

In [35]:
true_vals_0103 = pd.read_csv(os.path.join(true_val_location, 'true_counts_blk_0103.csv'))

In [36]:
true_vals_0103[['True_count']]

,True_count
0,40
1,39
2,41
3,31
4,32
5,40
6,27


In [37]:
mae_0103 = mean_absolute_error(true_vals_0103[['True_count']], frozen_preds_0103)
mae_0103

18.261184195022903